In [0]:
# =============================================================================
# NOTEBOOK  : silver_product_master
# PURPOSE   : bronze.product_master → silver.product_master
# LAYER     : Silver (SCD Type 2 — full version history)
# FREQUENCY : Weekly + incremental
# PATTERN   : spark.read + audit watermark
#             reads only new bronze rows since last successful run
#             no foreachBatch, no streaming checkpoint
#             watermark stored in audit table — fully transparent + controllable
#
# TRANSFORM:
#   - effective_date  : String → DateType
#   - expiry_date     : INT (days since epoch) → DateType
#   - cost_price      : Double → Decimal(10,2)
#   - selling_price   : Double → Decimal(10,2)
#   - dimensions      : "LxWxH" string → parsed into length, width, height
#   - scd_hash        : MD5 of all tracked business columns
#
# SCD TYPE 2 LOGIC (Option B — single atomic MERGE):
#   Tracked cols : product_name, category, subcategory, brand, supplier_id,
#                  cost_price, selling_price, weight, length, width, height, status
#   NEW     → INSERT as first version (valid_from=now, valid_to=null, is_current=True)
#   CHANGED → close old (is_current=False, valid_to=now) + insert new version
#   SAME    → IGNORE (scd_hash matches)
#
# REPROCESSING:
#   To reprocess from scratch → delete last SUCCESS entry from pipeline_audit
#   To reprocess from specific date → update end_time in pipeline_audit
#   No checkpoint files to manage
# =============================================================================
 
# ── CELL 1: Imports & Config ──────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
 
from utils.audit import start_audit, end_audit, get_last_successful_run_time
from utils.schema_registry import SILVER_PRODUCT_MASTER_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, md5, concat_ws,
    split, row_number
)
from pyspark.sql.types import (
    DecimalType, TimestampType, DoubleType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable
 
with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["bronze_product_master"]
TARGET_TABLE = cfg["tables"]["silver_product_master"]
PIPELINE     = "silver_product_master"
 
# Columns included in SCD2 hash — changes in any of these trigger new version
# product_id excluded (key), effective_date/expiry_date excluded (source metadata)
SCD_TRACKED_COLS = [
    "product_name", "category", "subcategory", "brand",
    "supplier_id", "cost_price", "selling_price",
    "weight", "length", "width", "height", "status"
]